# Sprawdzenie kernela 

Twoim silnikiem wykonawczym jest po prostu interpreter Pythona. A to oznacza, że zmienną kontekstu musisz utworzyć samodzielnie.

Zrób to w poniższym paragrafie. 

Przypomnij sobie, że w przypadku Spark SQL jest to obiekt klasy `SparkSession`.

Pamiętaj, zamiast importować ciężką bibliotekę `pyspark` możesz wykorzystać dostępne na Twojej maszynie środowisko *Apache Spark*. 
Ono już ma wszystko co potrzebujesz. Wystarczy, że skorzystasz z poniższego kodu 
```python
import findspark
findspark.init('/opt/spark')
``` 

In [32]:
import findspark
findspark.init('/opt/spark')

In [33]:
from pyspark.sql import SparkSession


Dowiedz się z jakim typem aplikacji masz do czynienia oraz jaki jest interfejs do sterownika.

In [35]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("jakasNazwa") \
    .getOrCreate()

spark

Dzięki powyższej informacji dowiedzieliśmy się nie tylko w jakim trybie został uruchomiony Spark obsługujący nasze polecenia, w jakiej jest wersji, ale także czy obsługuje funkcjonalność platformy Hive.

Dowiedz się także pod jakim użytkownikiem działamy w ramach tego notatnika.

In [36]:
%%sh 
whoami

hadoop


Czas na nasze właściwe zadania. 

W razie potrzeby korzystaj z https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

# 20 Years of Games

**7. Zaczytaj do zmiennej gameInfosDF zawartość pliku ign.csv**

In [37]:
username = "hadoop" # UWAGA! ustaw zmienną username na poprawną wartość

gameInfosDF=spark.read.\
    option("inferSchema", "true").\
    csv(f"/ign.csv", header=True).cache()

25/11/28 07:36:13 WARN CacheManager: Asked to cache already cached data.


**8. Wyświetl schemat zmiennej gameInfosDF**

In [38]:
gameInfosDF.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- score_phrase: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- score: double (nullable = true)
 |-- genre: string (nullable = true)
 |-- editors_choice: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_month: integer (nullable = true)
 |-- release_day: integer (nullable = true)



Możesz także po prostu przyglądnąć się jej kolumnom

In [22]:
gameInfosDF.columns

['_c0',
 'score_phrase',
 'title',
 'url',
 'platform',
 'score',
 'genre',
 'editors_choice',
 'release_year',
 'release_month',
 'release_day']

Zobaczmy też trzy pierwsze wiersze. Zróbmy to na kilka sposobów. 

* Na początek metoda `show()`

In [23]:
gameInfosDF.limit(3).show()

25/11/28 07:30:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , score_phrase, title, url, platform, score, genre, editors_choice, release_year, release_month, release_day
 Schema: _c0, score_phrase, title, url, platform, score, genre, editors_choice, release_year, release_month, release_day
Expected: _c0 but found: 
CSV file: hdfs://master:8020/ign.csv


+---+------------+--------------------+--------------------+----------------+-----+----------+--------------+------------+-------------+-----------+
|_c0|score_phrase|               title|                 url|        platform|score|     genre|editors_choice|release_year|release_month|release_day|
+---+------------+--------------------+--------------------+----------------+-----+----------+--------------+------------+-------------+-----------+
|  0|     Amazing|LittleBigPlanet P...|/games/littlebigp...|PlayStation Vita|  9.0|Platformer|             Y|        2012|            9|         12|
|  1|     Amazing|LittleBigPlanet P...|/games/littlebigp...|PlayStation Vita|  9.0|Platformer|             Y|        2012|            9|         12|
|  2|       Great|Splice: Tree of Life|/games/splice/ipa...|            iPad|  8.5|    Puzzle|             N|        2012|            9|         12|
+---+------------+--------------------+--------------------+----------------+-----+----------+------------

Przetwarzane dane mogą być duże. Wyniki natomiast z reguły są znacznie mniejsze, to pozwala nam je (o ile znamy ich wielkość) przekonwertować do obiektów `pandas DataFrame` i dzięki temu przedstawić w przyjaźniejszej postaci.
* metoda `toPandas()`

In [24]:
gameInfosDF.limit(3).toPandas()

,_c0,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
0,0,Amazing,LittleBigPlanet PS Vita,/games/littlebigplanet-vita/vita-98907,PlayStation Vita,9.0,Platformer,Y,2012,9,12
1,1,Amazing,LittleBigPlanet PS Vita -- Marvel Super Hero E...,/games/littlebigplanet-ps-vita-marvel-super-he...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
2,2,Great,Splice: Tree of Life,/games/splice/ipad-141070,iPad,8.5,Puzzle,N,2012,9,12


Za pomocą parametru konfiguracyjnego `spark.sql.repl.eagerEval.enabled` naszego kontekstu, również możemy 
ułatwić sobie wgląd w zawartość naszych wyników. Warto także ustawić parametr aby kontrolować liczbę pobieranych w ten sposób wierszy (tak, w razie niedoszacowania wyniku)
* parametr `spark.sql.repl.eagerEval.enabled`

In [26]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 3)
gameInfosDF

_c0,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
0,Amazing,LittleBigPlanet P...,/games/littlebigp...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
1,Amazing,LittleBigPlanet P...,/games/littlebigp...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
2,Great,Splice: Tree of Life,/games/splice/ipa...,iPad,8.5,Puzzle,N,2012,9,12


Wykorzystuj powyższe, aby móc podglądać uzyskiwane wyniki

 
**9. Na początek coś prostego. 
Wyświetl trzy najlepiej ocenione gry wydane w roku 2016 na platformę PC.**

In [55]:
from pyspark.sql.functions import col, lit
# tu wprowadź swoje rozwiazanie
gameInfosDF.filter((col("release_year")==2016) & (col("platform")=="PC")).orderBy(col("score").desc()).limit(3)


_c0,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
18417,Masterpiece,Undertale,/games/undertale/...,PC,10.0,RPG,Y,2016,1,13
18353,Masterpiece,The Witness,/games/the-witnes...,PC,10.0,Puzzle,Y,2016,1,25
18624,Masterpiece,Inside,/games/inside-pla...,PC,10.0,Adventure,Y,2016,6,28


**10. Określ dla każdej oceny opisowej (score_phrase) minimalną i 
maksymalną ocenę liczbową. Wyniki posortuj
rosnąco pod względem minimalnej oceny liczbowej.**

In [97]:
from pyspark.sql.functions import *
spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 20)
# tu wprowadź swoje rozwiazanie
gameInfosDF.groupBy("score_phrase").agg(min("score").alias("min_score"), max("score").alias("max_score")).sort(2)

score_phrase,min_score,max_score
Disaster,0.5,0.8
Unbearable,1.0,1.9
Painful,2.0,2.9
Awful,3.0,3.9
Bad,4.0,4.9
Mediocre,5.0,5.9
Okay,6.0,6.9
Good,7.0,7.9
Great,8.0,8.9
Amazing,9.0,9.9


**11. To może coś trudniejszego. Wyznacz liczbę oraz średnią ocenę gier wydawanych w poszczególnych latach
począwszy od roku 2000 na poszczególne platformy. Nie analizuj wszystkich platform – ogranicz je tylko do
tych, dla których liczba wszystkich recenzji gier biorąc pod uwagę wszystkie lata przekroczyła 500.**

*Uwaga: Klasycznie odwołalibyśmy się do źródłowego zboru danych dwa razy. Raz aby wyznaczyć popularne platformy, a następnie aby wyznaczyć ostateczny wynik. 
Korzystając z funkcji analitycznych możesz to zadanie rozwiązać sięgając do źródłowych danych tylko raz.*

**Rozwiąż to zadanie na dwa sposoby:**

a. Za pomocą DataFrame API


In [114]:
from pyspark.sql.window import Window
# tu wprowadź swoje rozwiazanie
w = Window.partitionBy("platform")
gameInfosDF.withColumn("number_of_reviews", count("score").over(w)) \
    .filter(col("release_year")>=2000) \
    .filter(col("number_of_reviews")>500) \
    .groupBy("platform", "release_year") \
    .agg(
        count("score"),
        avg("score")
    )


platform,release_year,count(score),avg(score)
Game Boy Advance,2001,76,6.803947368421052
Game Boy Advance,2002,172,6.693023255813954
Game Boy Advance,2003,141,6.858865248226948
Game Boy Advance,2004,105,6.570476190476191
Game Boy Advance,2005,56,6.601785714285714
Game Boy Advance,2006,61,5.88688524590164
Game Boy Advance,2007,12,6.191666666666666
GameCube,2001,17,8.152941176470588
GameCube,2002,132,6.969696969696967
GameCube,2003,137,6.86058394160584


b. Za pomocą SQL (po zarejestrowaniu źródeł danych jako tymczasowych perspektyw).

In [118]:
gameInfosDF.createOrReplaceTempView("gameinfos")
# tu wprowadź swoje rozwiazanie
result = spark.sql("""
WITH counted AS (
    SELECT
        *,
        COUNT(score) OVER (PARTITION BY platform) AS number_of_reviews
    FROM gamesinfos
)
SELECT
    platform,
    release_year,
    COUNT(score) AS count_score,
    AVG(score) AS avg_score
FROM counted
WHERE release_year >= 2000
  AND number_of_reviews > 500
GROUP BY platform, release_year
""")

result.show()



{"ts": "2025-11-28 08:16:58.876", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `gamesinfos` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o62.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `gamesinfos` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TA

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `gamesinfos` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 6 pos 9;
'Aggregate ['platform, 'release_year], ['platform, 'release_year, 'COUNT('score) AS count_score#17938, 'AVG('score) AS avg_score#17939]
+- 'Filter (('release_year >= 2000) AND ('number_of_reviews > 500))
   +- 'SubqueryAlias counted
      +- 'SubqueryAlias counted
         +- 'Project [*, 'COUNT('score) windowspecdefinition('platform, unspecifiedframe$()) AS number_of_reviews#17940]
            +- 'UnresolvedRelation [gamesinfos], [], false


**12. Jeśli masz swoją ulubioną serię gier (https://pl.wikipedia.org/wiki/Kategoria:Serie_gier_komputerowych)
zobacz jakie średnie oceny zdobyły poszczególne pozycje z tej serii. Wyniki posortuj chronologicznie.**

In [ ]:
# tu wprowadź swoje rozwiazanie


**13. (opcjonalne) Porównaj ze sobą gry wchodzące w skład wybranych serii gier wchodzących w skład 20
najlepszych serii wg Guinessa (lista z 2010 roku). W związku z tym, że gry nie są wydawane co roku, pogrupuj
dane w przedziały o długości 5 lat.**

In [ ]:
# tu wprowadź swoje rozwiazanie


In [ ]:
# brudnopis

 
# MondialDB – DataFrames

**14. Na początku do zmiennych `citiesDF`, `countriesDF` załaduj odpowiednio dane z plików
`mondial.cities.json`, `mondial.countries.json`**

In [ ]:
citiesDF = spark.read.json(f"/user/{username}/mondial.cities.json").cache()
countriesDF = spark.read.json(f"/user/{username}/mondial.countries.json").cache()

**15. Zapoznaj się z ich strukturą. Zwróć uwagę na występujące typy array.**

In [ ]:
citiesDF.printSchema()
countriesDF.printSchema()

**16. Zanim zaczniesz realizować zadania, zapoznaj się ze funkcją `explode`, która nadaje się świetnie do pracy z tablicami i ich rozpłaszczania.**

**Przykładowe zapytanie:**

In [ ]:
countriesDF.where("name = 'Poland'").\
            select(col("name"), explode(col("population")).alias("pop_in_years"))

Zwróć uwagę także na inne funkcje z tej rodziny jak: `explode_outer`, `posexplode`, `posexplode_outer`
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html


Wszystkie zadania wykonaj korzystając *DataFrame API*. Nie korzystaj z SQL.

**17. Oblicz sumę ludności wszystkich Państw na rok 2010. 
W sytuacji gdy w danym kraju nie przeprowadzono
badania w roku 2010 wykorzystaj najnowsze z badań wcześniejszych.**

In [ ]:
from pyspark.sql.window import Window
# tu wprowadź swoje rozwiazanie


**18. Było ciężko? Nie wierzę.**

**Teraz już będzie z górki. Podaj nazwy i gęstość zaludnienia trzech krajów o największej gęstości zaludnienia w roku 2010.**

In [ ]:
# tu wprowadź swoje rozwiazanie



**19. Podaj trzy kraje o największym procencie ludności żyjącym w miastach powyżej 50 000 mieszkańców w roku
2010.**

In [ ]:
# tu wprowadź swoje rozwiazanie


No cóż, dane dotyczące ludności w miastach są zapewne nowsze niż z 2010 roku.
Na marginesie, zarówno Melilla jak i Ceuta to hiszpańskie miasta, afrykańskie eksklawy położone na terytorium
Maroka. Oba liczą ponad 70 tyś mieszkańców i oba posiadają autonomię (uzyskaną jednocześnie w marcu 1995 roku)
dlatego znalazły się w naszym zestawieniu.
A co to takiego eksklawy i czy enklawa jest tym samym, to już możesz przeczytać samodzielnie np. tu:
https://pl.wikipedia.org/wiki/Eksklawa

In [ ]:
# brudnopis